# DATA CLEANING - ABS Income (Data by Region) by SA2

This notebook cleans the ABS **Income (Data by Region)** dataset down to **Victorian SA2 regions, 3 census years (2011, 2016, 2021), one income measure** - median equivalised total household income (weekly)

## Step 1 - Point to the file

In [38]:
import re

import pandas as pd

FILE_PATH = "../external_data/14100DO0004_2011-25.xlsx"
SHEET_NAME = "Table 1"
HEADER_ROW = 6

## Step 2 - Load the sheet
Loaded as text so ABS placeholders (`-`, `np`) don't get misread as numbers before we're ready for them.

In [39]:
raw_df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)
raw_df = raw_df.astype(str)
raw_df.head()

,Code,Label,Year,$1-$499 per week (%),$500-$999 per week (%),$1000-$1999 per week (%),$2000-$2999 per week (%),$3000 or more per week (%),Nil income (%),Negative income (%),...,Superannuation and annuity income as main source of income (%),Total income earners (excl. Government pensions and allowances) (no.),Total income earners (excl. Government pensions and allowances) - median age (years),Total income (excl. Government pensions and allowances) ($m),Median total income (excl. Government pensions and allowances) ($),Mean total income (excl. Government pensions and allowances) ($),Persons who reported gross capital gains (no.),Total value of gross capital gains ($m),Median value of gross capital gains ($),Mean value of gross capital gains ($)
0,AUS,Australia,2011.0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
1,AUS,Australia,2016.0,27.9,23.1,22.3,5.1,3.1,9,0.5,...,-,-,-,-,-,-,-,-,-,-
2,AUS,Australia,2017.0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,955098,50532.4,3654,52908
3,AUS,Australia,2018.0,-,-,-,-,-,-,-,...,-,-,-,-,-,-,991906,53892.1,3773,54332
4,AUS,Australia,2019.0,-,-,-,-,-,-,-,...,1.5,15744685,41,998968.8,49096,63448,950769,45220.3,2922,47562


## Step 3 - Only keep the columns we need

In [40]:
INCOME_COL = "Median equivalised total household income (weekly) ($)"

income_df = raw_df[["Code", "Label", "Year", INCOME_COL]].copy()

## Step 4 - Filter to Victorian SA2 codes
`Code` mixes Australia / State / SA4 / SA3 / SA2 / GCCSA in one column. SA2 codes are exactly 9 digits, and Victoria's state prefix is `2`.

In [41]:
is_vic_sa2 = income_df["Code"].str.match(r"^2\d{8}$")

before_rows = len(income_df)
income_df = income_df[is_vic_sa2]
print(f"{before_rows:,} rows -> {len(income_df):,} Victorian SA2 rows")

32,006 rows -> 5,742 Victorian SA2 rows


## Step 5 - Only keep 2011, 2016 and 2021
These are the only years with the data we need, per the Census schedule. Non-census years do not have this income measure reported.

In [42]:
CENSUS_YEARS = {2011, 2016, 2021}

income_df["Year"] = pd.to_numeric(income_df["Year"], errors="coerce")
before_rows = len(income_df)
income_df = income_df[income_df["Year"].isin(CENSUS_YEARS)]
print(f"{before_rows:,} rows -> {len(income_df):,} rows for census years")

5,742 rows -> 1,566 rows for census years


## Step 6 - Clean up column names
Make column names code-friendly.

In [43]:
def clean_col_name(name):
    """Lowercase, drop punctuation, replace spaces with underscores."""
    name = name.strip().lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name


income_df.columns = [clean_col_name(c) for c in income_df.columns]
print(income_df.columns.tolist())

['code', 'label', 'year', 'median_equivalised_total_household_income_weekly_']


## Step 7 - Cast types
`pd.to_numeric(errors="coerce")` turns ABS placeholders like `-` into `NaN` automatically, so no separate placeholder-mapping step is needed.

In [44]:
INCOME_COL_CLEAN = "median_equivalised_total_household_income_weekly_"

income_df["year"] = income_df["year"].astype("Int64")
income_df[INCOME_COL_CLEAN] = pd.to_numeric(
    income_df[INCOME_COL_CLEAN], errors="coerce"
)
income_df.dtypes


code                                                  object
label                                                 object
year                                                   Int64
median_equivalised_total_household_income_weekly_    float64
dtype: object

## Data quality checks

### Check 1 - Nulls per column

In [45]:
null_counts = income_df.isnull().sum()

print(f"Total rows: {len(income_df):,}")
print(null_counts)

Total rows: 1,566
code                                                  0
label                                                 0
year                                                  0
median_equivalised_total_household_income_weekly_    15
dtype: int64


### Drop rows with a null income
A handful of SA2/year combinations have no reported median income (ABS placeholder that became `NaN`). These rows aren't usable, so they're dropped.

In [46]:
before_rows = len(income_df)
income_df = income_df.dropna(subset=[INCOME_COL_CLEAN])
print(f"{before_rows:,} rows -> {len(income_df):,} rows ({before_rows - len(income_df):,} null-income rows dropped)")

1,566 rows -> 1,551 rows (15 null-income rows dropped)


### Check 2 - Each SA2 should have all 3 Census years, no duplicates
Checking row count alone can hide duplicates and missing years cancelling each other out, so this checks both row count **and** distinct year count per SA2.

In [47]:
sa2_summary = income_df.groupby("code")["year"].agg(
    row_count="count", year_count="nunique"
)
incomplete = sa2_summary[
    (sa2_summary["row_count"] != 3) | (sa2_summary["year_count"] != 3)
]

print(f"Distinct Victorian SA2 regions: {len(sa2_summary):,}")
print(f"Regions missing a year or with duplicate years: {len(incomplete):,}")
incomplete.head(20)

Distinct Victorian SA2 regions: 517
Regions missing a year or with duplicate years: 0


,row_count,year_count
code,,


### Check 3 - Implausible income values
Flags negative incomes or values above a sanity-check ceiling.

In [ ]:
INCOME_CEILING = 5000

suspect_rows = income_df[
    (income_df[INCOME_COL_CLEAN] < 0)
    | (income_df[INCOME_COL_CLEAN] > INCOME_CEILING)
]
print(f"Suspect income rows: {len(suspect_rows):,}")
suspect_rows

Suspect income rows: 0


,code,label,year,median_equivalised_total_household_income_weekly_


## Step 8 - Save the cleaned data
Exported as a single CSV file.

In [49]:
output_path = "../external_data/abs_income_sa2_vic_clean.csv"

income_df.to_csv(output_path, index=False)
print(f"Saved {len(income_df):,} rows to {output_path}.")

Saved 1,551 rows to ../external_data/abs_income_sa2_vic_clean.csv.
